# Proyecto Sprint 8 - Análisis de Viajes en Taxi en Chicago para Zuber

## 1. Introducción
Breve descripción del objetivo del proyecto y de las fuentes de datos utilizadas.

## 2. Paso 1: Extracción de Datos Meteorológicos

In [39]:
# Importar librerías
import requests # Importa la librería para enviar solicitudes al servidor
import pandas as pd # Importa la librería pandas con el alias "pd"
from bs4 import BeautifulSoup # Importa la librería para analizar la página web

# Almancenar la dirección del enlace en la variable "URL"
URL = 'https://practicum-content.s3.us-west-1.amazonaws.com/data-analyst-eng/moved_chicago_weather_2017.html'

# Guardar el objeto de respuesta en la variable requerida "req"
req = requests.get(URL) # Solicitud GET

# Crea un objeto BeautifulSoup y pasa el contenido de texto 
# de la solicitud GET y lo almacena en la variable "soup"
soup = BeautifulSoup(req.text, 'lxml')

# Aplica el método de búsqueda a la etiqueta de la tabla
# especificando el atributo de la tabla "weather_records"
# y lo almacena en la variable "table"
table = soup.find('table', attrs={'id': 'weather_records'})

# Crea un bucle para obtener los encabezados de la tabla 
# y los almacena en la lista "table_headers"
table_headers = []
for row in table.find_all('th'):
    table_headers.append(row.text)

# Crea un bucle para obtener los datos de la tabla 
# y los almacena en la lista "content"
content = []
for row in table.find('tbody').find_all('tr'):
    td_elements = row.find_all('td')
    if td_elements:
        content.append([td.text for td in td_elements])
 
# Crea un DataFrame de pandas llamado "weather_records"
weather_records = pd.DataFrame(content, columns=table_headers)

# Imprime el DataFrame
weather_records.head()


,Date and time,Temperature,Description
0,2017-11-01 00:00:00,276.150,broken clouds
1,2017-11-01 01:00:00,275.700,scattered clouds
2,2017-11-01 02:00:00,275.610,overcast clouds
3,2017-11-01 03:00:00,275.350,broken clouds
4,2017-11-01 04:00:00,275.240,broken clouds


## 2.2 Análisis exploratorio de datos meteorológicos

En esta seccion, se explorarán los datos del DataFrame usando los siguientes métodos para obtener una comprensión inicial:

- `sample()` para visualizar una muestra de los valores y su descripción.
- `info()` para conocer el tamaño del DataFrame, ver los tipos de datos e identificar valores nulos.
- `describe()` para obtener estadísticas generales.


Nos enfocaremos en identificar posibles problemas como:

- Valores faltantes.
- Valores duplicados.
- Tipos de datos incorrectos.
- Inconsistencias en los datos (uso inconsistente de mayúsculas, espacios innecesarios en texto, etc.).


Una vez inspeccionados los datos, se realizarán las correcciones necesarias para garantizar la calidad de los mismos antes del análisis.

In [40]:
# Vista previa de los datos:
print('Muestra aleatoria de los datos:')
weather_records.sample(10)

Muestra aleatoria de los datos:


,Date and time,Temperature,Description
488,2017-11-21 08:00:00,279.430,sky is clear
632,2017-11-27 08:00:00,275.610,sky is clear
392,2017-11-17 08:00:00,275.550,overcast clouds
36,2017-11-02 12:00:00,281.440,light rain
328,2017-11-14 16:00:00,277.230,mist
17,2017-11-01 17:00:00,278.320,overcast clouds
595,2017-11-25 19:00:00,282.580,sky is clear
442,2017-11-19 10:00:00,274.330,overcast clouds
85,2017-11-04 13:00:00,279.750,mist
522,2017-11-22 18:00:00,272.430,few clouds


In [41]:
# Imprime la información general/resumida sobre el DataFrame "weather_records"
print('Información general del Dataframe')
print()
print(weather_records.info())
print()
print()
print(f'Número de filas duplicadas: {weather_records.duplicated().sum()}')


Información general del Dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 697 entries, 0 to 696
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Date and time  697 non-null    object
 1   Temperature    697 non-null    object
 2   Description    697 non-null    object
dtypes: object(3)
memory usage: 16.5+ KB
None


Número de filas duplicadas: 0


### Observaciones del análisis preliminar

- El DataFrame `weather_records` tiene 697 registros y 3 columnas.
- No hay valores nulos ni filas duplicadas.
- Sin embargo, todas las columnas están en formato `object`, incluyendo la temperatura y la marca temporal.
- Las estadísticas descriptivas no son útiles mientras los datos estén en este formato.
- Es necesario convertir:
  - `Date and time` a tipo `datetime`
  - `Temperature` a tipo `float`
- También ### Observaciones del análisis preliminar

- El DataFrame `weather_records` tiene 697 registros y 3 columnas.
- No hay valores nulos ni filas duplicadas.
- Sin embargo, todas las columnas están en formato `object`, incluyendo la temperatura y la marca temporal.
- Las estadísticas descriptivas no son útiles mientras los datos estén en este formato.
- Es necesario convertir:
  - `Date and time` a tipo `datetime`
  - `Temperature` a tipo `float`
- Normalizaremos de nombres de las columnas:
    - Convertiremos todos los nombres de columnas a minúsculas.
    - Eliminaremos los espacios y los reemplazaremos por guiones bajos.
- También, aunque no se detectó ninguna anomalía al revisar la muestra aleatoria, se recomienda estandarizar la columna `Description` para detectar y prevenir duplicados implícitos (por ejemplo, "Rain" vs "rain", o espacios adicionales).

In [42]:
# Renombra las columnas para que estén todas en minúsculas y seguir las buenas prácticas
weather_records.columns = ['ts', 'temperature', 'description']

# Elimina espacios extras y convierte a minúsculas los valores en la columna "description"
weather_records['description'] = weather_records['description'].str.strip().str.lower()

# Convierte 'timestamp' a formato "datetime"
weather_records['ts'] = pd.to_datetime(weather_records['ts'])

# Convierte 'temperature' a "float"
weather_records['temperature'] = pd.to_numeric(weather_records['temperature'], errors='coerce')

In [43]:
# Verificar tipos de datos y valores nulos
print('Información general del Dataframe después de la corrección de los tipos de datos')
print()
weather_records.info()

Información general del Dataframe después de la corrección de los tipos de datos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 697 entries, 0 to 696
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ts           697 non-null    datetime64[ns]
 1   temperature  697 non-null    float64       
 2   description  697 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 16.5+ KB


In [44]:
# Verifica los nombres de las columnas después de haberlas normalizado
print('Nombres de las columnas después de la transformación:')
weather_records.columns

Nombres de las columnas después de la transformación:


Index(['ts', 'temperature', 'description'], dtype='object')

In [45]:
# Estadísticas descriptivas
print('Estadísticas descriptivas de la temperatura')
print()
print(weather_records['temperature'].describe())
print()
print()
# Número de valores únicos en la columna 'description'
print(f'Número de valores únicos en la columna description: {weather_records['description'].value_counts()}')
print()
print()
# Mostrar todos los valores únicos sin truncamiento
pd.set_option('display.max_rows', None)
print(weather_records['description'].unique())


Estadísticas descriptivas de la temperatura

count    697.000000
mean     277.548864
std        4.515403
min      265.740000
25%      274.240000
50%      277.410000
75%      280.980000
max      289.760000
Name: temperature, dtype: float64


Número de valores únicos en la columna description: description
sky is clear                        178
overcast clouds                     146
mist                                 97
broken clouds                        63
scattered clouds                     47
few clouds                           37
light rain                           31
fog                                  24
haze                                 18
light intensity drizzle              13
moderate rain                        12
light snow                           11
drizzle                               9
proximity thunderstorm                5
proximity thunderstorm with rain      2
thunderstorm with drizzle             1
thunderstorm with light rain          1
heavy intensity

### Análisis descriptivo de la temperatura y descripción del clima

- El DataFrame tiene 697 registros con temperaturas registradas en lo que parece ser grados **Kelvin**.
- La conversión de los extremos da como resultado:
  - Mínima: 265.74 K — -7.41 ºC
  - Máxima: 289.76 K — 16.61 ºC
- Esto concuerda con un clima otoñal en Chicago, donde es común experimentar tanto temperaturas bajo cero como temperaturas templadas en un mismo mes.
- La **media** y la **mediana** son muy similares (~277.5 K o 4.26 ºC), lo cual sugiere una distribución bastante simétrica o normal.
- A pesar de que el rango total entre el valor mínimo y el máximo abarca 22 K (Kelvin), la mayoría de los valores parecen estar concentrados cerca del promedio, lo que podría sugerir que las condiciones climáticas son relativamente estables dentro del mes.
- La descripción del clima muestra 19 categorías únicas. Algunas observaciones:
  - Las más frecuentes son `'sky is clear'`, `'overcast clouds'`, `'mist'`, `'broken clouds'`, `'scattered clouds'` y `'few clouds'` y `'light rain  '`.
  - Se identifican múltiples tipos de precipitación: `'light rain'`, `'moderate rain'`, `'drizzle'`, `'thunderstorm'`, `'light snow'` y `'heavy intensity rain'`.
  - No hay duplicados implícitos ni inconsistencias en la categorización.

> Con base en estos resultados, el siguiente paso será clasificar las condiciones climáticas en dos grupos: **"Good"** y **"Bad"** cuando realicemos el análisis exploratorio con SQL para facilitar el análisis de su posible relación con la duración de los viajes.


## 3. Paso 2: Análisis Exploratorio con SQL
- Consulta 1: Número de viajes por empresa (15-16 noviembre)
- Consulta 2: Viajes con “Yellow” o “Blue” (1-7 noviembre)
- Consulta 3: Flash Cab, Taxi Affiliation y "Other"
- Comentarios y observaciones

## 4. Paso 3: Preparación para Prueba de Hipótesis
- Identificación de barrios
- Clasificación del clima con CASE
- Viajes desde Loop a O’Hare los sábados
- Unión con tabla de clima

## 5. Paso 4: Análisis en Python

### 5.1 Carga y limpieza de `project_sql_result_01.csv`
### 5.2 Carga y limpieza de `project_sql_result_04.csv`

### 5.3 Gráficos
- Empresas vs número de viajes
- Barrios con más finalizaciones
- Conclusiones

## 6. Paso 5: Prueba de Hipótesis en Python
- Hipótesis
- Método de prueba
- Valor de alfa
- Resultados y conclusiones

## 7. Conclusiones Generales
Resumen de hallazgos, implicaciones para Zuber y recomendaciones.

## 8. Comentarios Adicionales (opcional)
Reflexiones personales, posibles mejoras o análisis futuros.